# CHSH

**Contributed by**: Marek Kaluba and Benoît Legat
**Adapted from**: [Talk](https://jump.dev/assets/jump-dev-workshops/2024/legat.html) at [JuMP-dev 2024](https://jump.dev/meetings/jumpdev2024/)

The goal of this tutorial is to show how to use SumOfSquares.jl with a custom algebra that is **not** defined
with [DynamicPolynomials](https://github.com/JuliaAlgebra/DynamicPolynomials.jl/) or
[TypedPolynomials](https://github.com/JuliaAlgebra/TypedPolynomials.jl/).
Even though some part of SumOfSquares.jl only works for monomials defined by these packages,
there is an effort to abstract away as much as possible on top of
[StarAlgebras](https://github.com/JuliaAlgebra/StarAlgebras.jl/).

This illustrate this, in this tutorial, we are doing to define a custom monoid structure implementing the
rewriting rules of the [CHSH inequality](https://en.wikipedia.org/wiki/CHSH_inequality) using
[KnuthBendix](https://github.com/kalmarek/KnuthBendix.jl).

In [1]:
using SumOfSquares
include(
    joinpath(dirname(dirname(pathof(SumOfSquares))), "examples", "monoids.jl"),
)

import StarAlgebras as SA
import KnuthBendix as KB
import GroupsCore

The rewriting rule are as follows:

In [2]:
monoid, A, C = trace_monoid(2, 2, A = :A, C = :C)
monoid.rws

reduced, confluent rewriting system with 8 active rules.
rewriting ordering: LenLex: A1 < A2 < C1 < C2
┌──────┬──────────────────────────────────┬──────────────────────────────────┐
│ Rule │                              lhs │ rhs                              │
├──────┼──────────────────────────────────┼──────────────────────────────────┤
│    1 │                             A1^2 │ (id)                             │
│    2 │                             A2^2 │ (id)                             │
│    3 │                            C1*A1 │ A1*C1                            │
│    4 │                            C1*A2 │ A2*C1                            │
│    5 │                             C1^2 │ (id)                             │
│    6 │                            C2*A1 │ A1*C2                            │
│    7 │                            C2*A2 │ A2*C2                            │
│    8 │                             C2^2 │ (id)                             │
└──────┴────────────────────

We now define a `StarAlgebra` from [StarAlgebras](https://github.com/JuliaAlgebra/StarAlgebras.jl/)

In [3]:
RM = let monoid = monoid, A = A, C = C, level = 4
    A_l, sizesA = Monoids.wlmetric_ball(A, radius = level)
    C_l, sizesC = Monoids.wlmetric_ball(C, radius = level)
    @time words, sizes = Monoids.wlmetric_ball(
        unique!([a * c for a in A_l for c in C_l]);
        radius = 2,
    )
    @info "Sizes of generated balls:" (A, C, combined) =
        (sizesA, sizesC, sizes)
    basis = SA.SubBasis(SA.DiracBasis(monoid), words)
    dirac = SA.DiracMStructure(basis, *)
    table = SA.MTable(dirac, (sizes[1], sizes[1]))
    SA.StarAlgebra(monoid, table)
end

  0.103488 seconds (167.63 k allocations: 7.952 MiB, 95.36% compilation time)
┌ Info: Sizes of generated balls:
└   (A, C, combined) = ([3, 5, 7, 9], [3, 5, 7, 9], [81, 289])


*-algebra of monoid with 8 relations over Alphabet{Symbol}: [:A1, :A2, :C1, :C2]

We can convert a monoid element:

In [4]:
A[1], typeof(A[1])

(A1, Main.var"##699".Monoids.MonoidElement{UInt16, Main.var"##699".Monoids.Monoid{UInt16, KnuthBendix.Alphabet{Symbol}, KnuthBendix.RewritingSystem{KnuthBendix.Words.Word{UInt16}, KnuthBendix.LenLex{Symbol}}}})

to an element of the algebra `RM` (so essentially `1 ⋅ A`) as follows:

In [5]:
RM(A[1])

1·A1

Then, we can do arithmetic on these algebra element to form the CHSH inequality:

In [6]:
chsh = let A = RM.(A), C = RM.(C)
    A[1] * C[1] + A[1] * C[2] + A[2] * C[1] - A[2] * C[2]
end

1·A1·C1 + 1·A1·C2 + 1·A2·C1 - 1·A2·C2

Currently, SumOfSquares only store the basis and it uses `MB.algebra` to
obtain an algebra from the basis so we need to implement `MB.algebra`.

In [7]:
import MultivariateBases as MB
function MB.algebra(b::SA.DiracBasis{<:Monoids.MonoidElement})
    return SA.StarAlgebra(SA.object(b), b)
end

f = SA.AlgebraElement(
    SA.SparseCoefficients(
        [SA.basis(chsh)[k] for (k, _) in SA.nonzero_pairs(SA.coeffs(chsh))],
        [v for (_, v) in SA.nonzero_pairs(SA.coeffs(chsh))],
    ),
    MB.algebra(parent(SA.basis(chsh))),
)

1·A1·C1 + 1·A1·C2 + 1·A2·C1 - 1·A2·C2

We pick the SCS solver:

In [8]:
import SCS
scs = optimizer_with_attributes(
    SCS.Optimizer,
    "eps_abs" => 1e-9,
    "eps_rel" => 1e-9,
    "max_iters" => 1000,
)

MathOptInterface.OptimizerWithAttributes(SCS.Optimizer, Pair{MathOptInterface.AbstractOptimizerAttribute, Any}[MathOptInterface.RawOptimizerAttribute("eps_abs") => 1.0e-9, MathOptInterface.RawOptimizerAttribute("eps_rel") => 1.0e-9, MathOptInterface.RawOptimizerAttribute("max_iters") => 1000])

We currently need to manually add all SumOfSquares bridges as follows:

In [9]:
model = Model(scs)
SumOfSquares.Bridges.add_all_bridges(backend(model).optimizer, Float64)
@variable(model, λ)
@objective(model, Min, λ)

λ

Adding the SumOfSquares constraint is currently not as user-friendly than
with monomials:

In [10]:
n = size(SA.mstructure(RM).table, 1)
gram_basis = SA.sub_basis(SA.basis(chsh), 1:n)
cone = SumOfSquares.WeightedSOSCone{MOI.PositiveSemidefiniteConeTriangle}(
    SA.basis(chsh),
    [gram_basis],
    [one(f)],
)
@constraint(model, SA.coeffs(λ * one(f) - f, SA.basis(chsh)) in cone);

optimize!(model)
solution_summary(model)
objective_value(model) ≈ 2√2

------------------------------------------------------------------
	       SCS v3.2.11 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
------------------------------------------------------------------
problem:  variables n: 3051, constraints m: 3339
cones: 	  z: primal zero / dual free vars: 18
	  s: psd vars: 3321, ssize: 1
settings: eps_abs: 1.0e-09, eps_rel: 1.0e-09, eps_infeas: 1.0e-07
	  alpha: 1.50, scale: 1.00e-01, adaptive_scale: 1
	  max_iters: 1000, normalize: 1, rho_x: 1.00e-06
	  acceleration_lookback: 10, acceleration_interval: 10
	  compiled with openmp parallelization enabled
lin-sys:  sparse-direct-amd-qdldl
	  nnz(A): 6101, nnz(P): 0
------------------------------------------------------------------
 iter | pri res | dua res |   gap   |   obj   |  scale  | time (s)
------------------------------------------------------------------
     0| 3.30e+01  9.98e-01  2.67e+03 -1.34e+03  1.00e-01  5.72e-03 
   250| 1.06e-07  4.47e-06  8.60e-05  2.83e

true

Let's look at the size of the generated SDP:

In [11]:
mutable struct SDPSize
    free::Int
    eq::Vector{Int}
    psd::Vector{Int}
end
function Base.show(io::IO, size::SDPSize)
    print(io, "$(size.free) free variables, ")
    print(io, "$(sum(size.eq, init = 0)) equality constraints, ")
    print(io, "PSD block sizes: $(size.psd)")
    return
end
function _add!(f, psd, model, F, S)
    append!(
        psd,
        [
            f(MOI.get(model, MOI.ConstraintSet(), ci)) for
            ci in MOI.get(model, MOI.ListOfConstraintIndices{F,S}())
        ],
    )
    return
end
function summary(model)
    size = SDPSize(0, Int[], Int[])
    size.free = MOI.get(model, MOI.NumberOfVariables())
    F = MOI.VectorOfVariables
    S = MOI.PositiveSemidefiniteConeTriangle
    _add!(MOI.side_dimension, size.psd, model, F, S)
    S = SCS.ScaledPSDCone
    _add!(MOI.side_dimension, size.psd, model, F, S)
    size.free -= sum(size.psd, init = 0) do d
        return div(d * (d + 1), 2)
    end
    F = MOI.VectorAffineFunction{Float64}
    S = MOI.PositiveSemidefiniteConeTriangle
    _add!(MOI.side_dimension, size.psd, model, F, S)
    S = SCS.ScaledPSDCone
    _add!(MOI.side_dimension, size.psd, model, F, S)
    eq = Int[]
    F = MOI.VectorAffineFunction{Float64}
    S = MOI.Zeros
    _add!(MOI.dimension, size.eq, model, F, S)
    F = MOI.ScalarAffineFunction{Float64}
    S = MOI.EqualTo{Float64}
    _add!(MOI.dimension, size.eq, model, F, S)
    return size
end
summary(backend(model).optimizer.model)

3051 free variables, 18 equality constraints, PSD block sizes: [81]

We can see that the `FreeBridge` was used because SCS supports free variables and
affine-in-PSD constraints.

In [12]:
print_active_bridges(model)

 * Unsupported objective: MOI.VariableIndex
 |  bridged by:
 |   MOIB.Objective.FunctionConversionBridge{Float64, MOI.ScalarAffineFunction{Float64}, MOI.VariableIndex}
 |  may introduce:
 |   * Supported objective: MOI.ScalarAffineFunction{Float64}
 * Unsupported constraint: MOI.VectorAffineFunction{Float64}-in-SumOfSquares.WeightedSOSCone{MOI.PositiveSemidefiniteConeTriangle, StarAlgebras.SubBasis{Main.var"##699".Monoids.MonoidElement{UInt16, Main.var"##699".Monoids.Monoid{UInt16, KnuthBendix.Alphabet{Symbol}, KnuthBendix.RewritingSystem{KnuthBendix.Words.Word{UInt16}, KnuthBendix.LenLex{Symbol}}}}, Int64, Main.var"##699".Monoids.MonoidElement{UInt16, Main.var"##699".Monoids.Monoid{UInt16, KnuthBendix.Alphabet{Symbol}, KnuthBendix.RewritingSystem{KnuthBendix.Words.Word{UInt16}, KnuthBendix.LenLex{Symbol}}}}, StarAlgebras.DiracBasis{Main.var"##699".Monoids.MonoidElement{UInt16, Main.var"##699".Monoids.Monoid{UInt16, KnuthBendix.Alphabet{Symbol}, KnuthBendix.RewritingSystem{KnuthBendix.

As a workaround for [this MOI issue](https://github.com/jump-dev/MathOptInterface.jl/pull/3001),
we need to remove the Image bridge

In [13]:
import Dualization
model = Model(Dualization.dual_optimizer(scs))
SumOfSquares.Bridges.add_all_bridges(backend(model).optimizer, Float64)
@variable(model, λ)
@objective(model, Min, λ)
@constraint(model, SA.coeffs(λ * one(f) - f, SA.basis(chsh)) in cone);
optimize!(model)
solution_summary(model)
objective_value(model) ≈ 2√2

------------------------------------------------------------------
	       SCS v3.2.11 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
------------------------------------------------------------------
problem:  variables n: 289, constraints m: 3322
cones: 	  z: primal zero / dual free vars: 1
	  s: psd vars: 3321, ssize: 1
settings: eps_abs: 1.0e-09, eps_rel: 1.0e-09, eps_infeas: 1.0e-07
	  alpha: 1.50, scale: 1.00e-01, adaptive_scale: 1
	  max_iters: 1000, normalize: 1, rho_x: 1.00e-06
	  acceleration_lookback: 10, acceleration_interval: 10
	  compiled with openmp parallelization enabled
lin-sys:  sparse-direct-amd-qdldl
	  nnz(A): 5402, nnz(P): 0
------------------------------------------------------------------
 iter | pri res | dua res |   gap   |   obj   |  scale  | time (s)
------------------------------------------------------------------
     0| 9.13e-01  8.73e+00  1.09e+01  3.25e+00  1.00e-01  4.51e-03 
   250| 7.90e-09  4.67e-09  1.05e-07 -2.83e+0

true

The model is much smaller this time, with 289 equality constraints and 1 free variable.

In [14]:
summary(backend(model).optimizer.model)

1 free variables, 289 equality constraints, PSD block sizes: [81]

After dualization that is 289 free variables and 1 equality constraints.
This is a big improvement compared to the 3051 free variables, 18 equality constraints without dualization.

In [15]:
summary(backend(model).optimizer.model.optimizer.dual_problem.dual_model)

289 free variables, 1 equality constraints, PSD block sizes: [81]

We can see that the bridge used is the `KernelBridge` this time.

In [16]:
print_active_bridges(model)

 * Supported objective: MOI.VariableIndex
 * Unsupported constraint: MOI.VectorAffineFunction{Float64}-in-SumOfSquares.WeightedSOSCone{MOI.PositiveSemidefiniteConeTriangle, StarAlgebras.SubBasis{Main.var"##699".Monoids.MonoidElement{UInt16, Main.var"##699".Monoids.Monoid{UInt16, KnuthBendix.Alphabet{Symbol}, KnuthBendix.RewritingSystem{KnuthBendix.Words.Word{UInt16}, KnuthBendix.LenLex{Symbol}}}}, Int64, Main.var"##699".Monoids.MonoidElement{UInt16, Main.var"##699".Monoids.Monoid{UInt16, KnuthBendix.Alphabet{Symbol}, KnuthBendix.RewritingSystem{KnuthBendix.Words.Word{UInt16}, KnuthBendix.LenLex{Symbol}}}}, StarAlgebras.DiracBasis{Main.var"##699".Monoids.MonoidElement{UInt16, Main.var"##699".Monoids.Monoid{UInt16, KnuthBendix.Alphabet{Symbol}, KnuthBendix.RewritingSystem{KnuthBendix.Words.Word{UInt16}, KnuthBendix.LenLex{Symbol}}}}, Main.var"##699".Monoids.Monoid{UInt16, KnuthBendix.Alphabet{Symbol}, KnuthBendix.RewritingSystem{KnuthBendix.Words.Word{UInt16}, KnuthBendix.LenLex{Symbol}}

We didn't test using the `ImageBridge` with dualization or using the `KernelBridge` without dualization.
However, these will always produce larger models, which is why the choice o bridge can be done
automatically once you choose whether you want to dualize the problem or not.

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*